# Data Explorer: The Four Real Datasets

A short, standalone exploratory pass over the four real (non-synthetic) datasets used in the study -- PovertyMap-WILDS and the three real-world tabular datasets (housing, socio-economic, air quality) described in Section 3.6 ("Data sources") of the main text. This notebook does not re-run any SSL method or the mismatch-severity sweep; it only loads each dataset once, describes its covariates, and produces a small set of figures meant to give a reader quick, direct insight into what the underlying data actually look like, to support a brief "Data" section in the supplementary material.

Sections:
1. Load all four datasets
2. Size, dimensionality, and label-balance summary
3. Covariate summary statistics
4. Missingness check
5. Feature separability by label

In [5]:
pip install matplotlib

  Using cached matplotlib-3.11.1-cp311-cp311-win_amd64.whl.metadata (80 kB)
  Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp311-cp311-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp311-cp311-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp311-cp311-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.11.1-cp311-cp311-win_amd64.whl (9.3 MB)
Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl (225 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.63.0-cp311-cp311-win_amd64.whl (2.4 MB)
Using cached kiwisolver-1.5.0-cp311-cp311-win_amd64.whl (73 kB)
Using cached pillow-12.3.0-cp311-cp311-win_amd64.whl (7.2 MB)
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
Note: you may need to restart the kernel to use updated packages.

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ssl_spatial.data.housing import load_housing, FEATURE_NAMES as HOUSING_FEATURES
from ssl_spatial.data.socioeconomic import load_socioeconomic, FEATURE_NAMES as SOCIO_FEATURES
from ssl_spatial.data.air_quality import load_air_quality, FEATURE_NAMES as AIR_FEATURES
from ssl_spatial.data.wilds_poverty import (
    build_feature_matrix, build_pools, download_metadata, all_pool_indices,
)
from ssl_spatial.plotting import (
    BASELINE_COLOR, FONT_LABEL, FONT_LEGEND, FONT_TICK, FONT_TITLE, GRIDLINE,
    INK_MUTED, INK_PRIMARY, INK_SECONDARY, PALETTE, SURFACE,
    new_figure, panel_label, style_axes, savefig,
)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

REAL_WORLD_DIR = REPO_ROOT / "data" / "real_world"
WILDS_DIR = REPO_ROOT / "data" / "wilds_povertymap"
# Figures go straight into the manuscript's own figures folder (not the repo-root
# figures/), since that's the path manuscript/main/supplementary.tex actually reads
# via \graphicspath{{figures/}} -- a top-level figures/ folder is never compiled.
FIG_DIR = REPO_ROOT / "manuscript" / "main" / "figures"
FIG_DIR.mkdir(exist_ok=True, parents=True)
print("Repo root:", REPO_ROOT)

## 1. Load all four datasets

The three tabular datasets load directly through their existing `ssl_spatial.data` loaders. PovertyMap-WILDS is reassembled the same way the benchmark runners do: the official DHS cluster metadata plus the fixed labelled/unlabelled/test pools (`build_pools`, default fold A settings matching `configs/wilds_benchmark.yaml`), with per-cluster image features taken from the pre-built cache in `data/wilds_povertymap/` when present, falling back to extracting them from the cached `.npz` image patches otherwise. Unlike the modelling pipeline, no PCA reduction is applied here -- the raw 40 band-summary features (mean, std, p10, p50, p90 for each of 8 Landsat/nightlight bands) are kept so the covariates below are directly interpretable.

In [2]:
housing_df = load_housing(REAL_WORLD_DIR / "housing")
socio_df = load_socioeconomic(REAL_WORLD_DIR / "socioeconomic")
air_df = load_air_quality(REAL_WORLD_DIR / "air_quality", REAL_WORLD_DIR / "socioeconomic")

print(f"housing:       {housing_df.shape[0]:>6} rows, {housing_df['region'].nunique():>3} regions (CA counties), "
      f"{len(HOUSING_FEATURES)} covariates")
print(f"socioeconomic: {socio_df.shape[0]:>6} rows, {socio_df['region'].nunique():>3} regions (US states), "
      f"{len(SOCIO_FEATURES)} covariates")
print(f"air_quality:   {air_df.shape[0]:>6} rows, {air_df['region'].nunique():>3} regions (US states), "
      f"{len(AIR_FEATURES)} covariates")

housing:        20640 rows,  58 regions (CA counties), 6 covariates
socioeconomic:   3130 rows,  50 regions (US states), 4 covariates
air_quality:      855 rows,  50 regions (US states), 3 covariates


In [3]:
metadata = download_metadata(WILDS_DIR).reset_index(drop=True)
pools = build_pools(metadata)  # defaults match configs/wilds_benchmark.yaml (fold A, cpc=90, u=300, ti=150, to=150, seed=0)
needed = all_pool_indices(pools)

cache_path = WILDS_DIR / "feature_cache_foldA_cpc90_u300_ti150_to150_ps0.npz"
if cache_path.exists():
    cached = np.load(cache_path)
    X_cache, valid_idx = cached["X_cache"], cached["valid_idx"]
    missing = [i for i in needed if i not in set(valid_idx.tolist())]
    if missing:
        raise FileNotFoundError(f"{len(missing)}/{len(needed)} pool indices missing from cache {cache_path}")
    print(f"Loaded cached WILDS features: {X_cache.shape} from {cache_path.name}")
else:
    X_cache, valid_idx = build_feature_matrix(needed, WILDS_DIR)
    print(f"Built WILDS features from cached image patches: {X_cache.shape}")

from ssl_spatial.data.wilds_poverty import FEATURE_NAMES as WILDS_BAND_FEATURES

wilds_df = pd.DataFrame(X_cache, columns=WILDS_BAND_FEATURES)
wilds_df["lat"] = metadata.loc[valid_idx, "lat"].to_numpy()
wilds_df["lon"] = metadata.loc[valid_idx, "lon"].to_numpy()
wilds_df["region"] = metadata.loc[valid_idx, "country"].to_numpy()
wealth = metadata.loc[valid_idx, "wealthpooled"].to_numpy()
wilds_df["label"] = (wealth > metadata["wealthpooled"].median()).astype(int)

# For readability below, use one representative summary statistic (the per-band
# mean) per spectral/nightlight band as the "covariates" of interest, rather than
# all 40 mean/std/percentile features.
WILDS_MEAN_FEATURES = [c for c in WILDS_BAND_FEATURES if c.endswith("_mean")]

print(f"wilds_povertymap: {wilds_df.shape[0]:>6} rows, {wilds_df['region'].nunique():>3} regions (countries), "
      f"{len(WILDS_BAND_FEATURES)} raw band-summary covariates ({len(WILDS_MEAN_FEATURES)} per-band means)")

Loaded cached WILDS features: (1770, 40) from feature_cache_foldA_cpc90_u300_ti150_to150_ps0.npz
wilds_povertymap:   1770 rows,  18 regions (countries), 40 raw band-summary covariates (8 per-band means)


## 2. Size, dimensionality, and label-balance summary

Every label across all four datasets is binarised at its own median (see Section 3.6 of the main text), so an approximately 50/50 split is expected by construction; the table below checks that this holds and reports how many geographic regions (the unit the mismatch-severity sweep restricts) each dataset spans.

In [4]:
datasets = {
    "PovertyMap-WILDS": (wilds_df, WILDS_MEAN_FEATURES, "country"),
    "Housing": (housing_df, HOUSING_FEATURES, "CA county"),
    "Socio-economic": (socio_df, SOCIO_FEATURES, "US state"),
    "Air quality": (air_df, AIR_FEATURES, "US state"),
}

summary = pd.DataFrame([
    {
        "dataset": name,
        "n_rows": df.shape[0],
        "n_regions": df["region"].nunique(),
        "region_unit": unit,
        "n_covariates": len(feats),
        "pct_label_1": round(100 * df["label"].mean(), 1),
    }
    for name, (df, feats, unit) in datasets.items()
])
summary

,dataset,n_rows,n_regions,region_unit,n_covariates,pct_label_1
0,PovertyMap-WILDS,1770,18,country,8,53.7
1,Housing,20640,58,CA county,6,50.0
2,Socio-economic,3130,50,US state,4,49.5
3,Air quality,855,50,US state,3,49.9


## 3. Covariate summary statistics

Per-dataset descriptive statistics for each covariate used by the SSL methods (units follow each source: e.g. `MedInc` is tens of thousands of USD, `log_median_income`/`log_population` are natural-log scale, band means are calibrated surface-reflectance/DN units for PovertyMap-WILDS).

In [5]:
for name, (df, feats, _unit) in datasets.items():
    print(f"--- {name} ---")
    display(df[feats].describe().T.round(3))

--- PovertyMap-WILDS ---


,count,mean,std,min,25%,50%,75%,max
BLUE_mean,1770.0,-0.028,0.834,-1.468,-0.608,-0.202,0.368,2.982
GREEN_mean,1770.0,-0.040,0.827,-1.740,-0.591,-0.248,0.281,3.721
RED_mean,1770.0,-0.035,0.845,-1.558,-0.622,-0.186,0.327,3.760
SWIR1_mean,1770.0,-0.012,0.808,-2.511,-0.534,-0.157,0.299,3.379
SWIR2_mean,1770.0,-0.001,0.843,-1.852,-0.612,-0.121,0.409,3.708
TEMP1_mean,1770.0,-0.076,0.840,-2.675,-0.648,-0.200,0.385,2.486
NIR_mean,1770.0,-0.018,0.751,-3.544,-0.479,-0.088,0.380,3.031
NIGHTLIGHTS_mean,1770.0,0.248,1.234,-0.229,-0.216,-0.174,0.020,17.602


--- Housing ---


,count,mean,std,min,25%,50%,75%,max
MedInc,20640.0,3.871,1.900,0.500,2.563,3.535,4.743,15.000
HouseAge,20640.0,28.639,12.586,1.000,18.000,29.000,37.000,52.000
AveRooms,20640.0,5.429,2.474,0.846,4.441,5.229,6.052,141.909
AveBedrms,20640.0,1.097,0.474,0.333,1.006,1.049,1.100,34.067
Population,20640.0,1425.477,1132.462,3.000,787.000,1166.000,1725.000,35682.000
AveOccup,20640.0,3.071,10.386,0.692,2.430,2.818,3.282,1243.333


--- Socio-economic ---


,count,mean,std,min,25%,50%,75%,max
unemployment_rate,3130.0,3.584,1.203,0.300,2.800,3.400,4.200,17.300
log_median_income,3130.0,11.025,0.239,10.274,10.868,11.015,11.163,12.029
log_population,3130.0,10.273,1.515,3.761,9.284,10.164,11.139,16.084
pct_bachelors,3130.0,24.045,10.157,0.000,16.999,21.423,28.606,79.726


--- Air quality ---


,count,mean,std,min,25%,50%,75%,max
ozone_mean,855.0,0.044,0.004,0.023,0.044,0.045,0.045,0.056
pm10_mean,855.0,19.440,5.704,5.029,18.719,18.719,18.719,75.974
log_county_population,855.0,12.489,1.594,6.941,11.499,12.624,13.629,16.084


## 4. Missingness check

Each loader already drops rows with missing values in the fields it constructs, so this is a confirmation rather than a discovery step.

In [6]:
for name, (df, feats, _unit) in datasets.items():
    n_missing = df[feats + ["lat", "lon", "region", "label"]].isna().sum().sum()
    print(f"{name:<20} missing values: {n_missing}")

PovertyMap-WILDS     missing values: 0
Housing              missing values: 0
Socio-economic       missing values: 0
Air quality          missing values: 0


## 5. Feature separability by label

A boxplot per covariate, split by the binary label (below/above median), for each dataset, saved as its own standalone figure at standard font sizes -- since PovertyMap-WILDS has more than twice as many covariates as Air quality, forcing all four onto one shared panel size (to keep them visually comparable side by side) came at the cost of legibility; each dataset now gets its own figure, sized independently to its own content. Since PCA is applied to PovertyMap-WILDS's raw band features before modelling (Section 3.6 of the main text) but not to the three tabular datasets, a raw-covariate correlation matrix would not reflect what the classifiers actually see; this view instead shows directly how much each covariate separates the two label classes, which is the more relevant summary of the modelling task's difficulty. Each covariate's separation is also summarised by Cohen's $d$ (a standardised mean difference) for reference in the discussion.</cell id="ce147420">


In [ ]:
below_color, above_color = PALETTE[0], PALETTE[3]

def cohens_d(df, feat):
    below = df.loc[df["label"] == 0, feat]
    above = df.loc[df["label"] == 1, feat]
    pooled_sd = np.sqrt((below.var() + above.var()) / 2)
    return (above.mean() - below.mean()) / pooled_sd

# Report every covariate's effect size for reference in the discussion, but plot
# all covariates -- dropping any risks hiding a real, if modest, difference.
for name, (df, feats, unit) in datasets.items():
    effects = {feat: cohens_d(df, feat) for feat in feats}
    print(f"{name:<18}", {f: round(d, 2) for f, d in effects.items()})

dataset_filenames = {
    "PovertyMap-WILDS": "wilds",
    "Housing": "housing",
    "Socio-economic": "socioeconomic",
    "Air quality": "airquality",
}

# Each dataset is its own standalone figure at standard font sizes (no cross-panel
# scaling), since forcing PovertyMap-WILDS's 8 covariates and Air quality's 3 onto
# a shared panel size only trades one dataset's legibility for another's.
for name, (df, feats, _unit) in datasets.items():
    fig, axes = new_figure(figsize=(2.6 * len(feats), 3.2), ncols=len(feats))
    axes = np.atleast_1d(axes)
    for ax, feat in zip(axes, feats):
        below = df.loc[df["label"] == 0, feat].to_numpy()
        above = df.loc[df["label"] == 1, feat].to_numpy()
        bp = ax.boxplot([below, above], tick_labels=["Below\nmedian", "Above\nmedian"],
                         patch_artist=True, widths=0.6,
                         medianprops=dict(color=INK_PRIMARY, linewidth=1.5))
        for patch, color in zip(bp["boxes"], (below_color, above_color)):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
            patch.set_edgecolor(INK_SECONDARY)
        for element in ("whiskers", "caps"):
            for line in bp[element]:
                line.set_color(INK_MUTED)
        ax.tick_params(axis="x", labelsize=FONT_TICK - 2)
        ax.set_title(feat, fontsize=FONT_TITLE - 2, color=INK_PRIMARY)
    fig.suptitle(name, fontsize=FONT_TITLE, color=INK_PRIMARY, y=1.05)
    savefig(fig, FIG_DIR / f"data_explorer_box_{dataset_filenames[name]}.png")
    plt.show()